# Essig context → FCOs → vision model, plus novelty & filter test case
Each Essig source is its own FCO (content-addressed; Merkle root = id). They hybridize at a break point into one
**specimen-context FCO** that grounds the vision model, flags collection-gap taxa as novel, and lists what's still missing.
Then a test case: novel sizes → mesh gaps → filter suggestions with preservation risk → filters × hardware × capture
explored in a Merkle ledger, with LOMO compared against break points.

**Swap data in the next cell.** `gaps.html` and the essigdb home page disallow automated fetching, so save them from your browser.

In [1]:
# ---- swap your data here ----
SAVED_HOME_HTML = None      # e.g. "../data/essigdb_home.html"  (saved from browser)
SAVED_GAPS_HTML = None      # e.g. "../data/essig_gaps.html"    (saved from browser)
LABEL_TEXT = None           # text from eme_label output for one specimen (None = example)
GBIF_RECORD_JSON = None     # one GBIF occurrence JSON for that EMEC number (None = example)
LABEL_PHOTO = None          # photo of the pinned label -> Claude reads it (needs ANTHROPIC_API_KEY)
SAMPLE_CSV = None           # length_mm,width_mm,profile for novelty test (None = built-in bag)
PRESERVATION = "ethanol_70"
SEED = 7

In [2]:
import sys, json, itertools, numpy as np, pandas as pd, matplotlib.pyplot as plt, cv2
sys.path.insert(0, "..")
from baebugs.essig_adapter import page_fco, label_fco, gbif_fco, vision_label_fco, read_label_with_claude, specimen_context, vision_prompt_context
from baebugs.provenance import sha256_bytes
home = page_fco("home", SAVED_HOME_HTML)
gaps = page_fco("gaps", SAVED_GAPS_HTML)
if not SAVED_GAPS_HTML:   # example gap list so the lane runs offline; replace with the saved page
    from baebugs.fco import FCO
    gaps = FCO("essig_page:gaps", gaps.source, {"gap_items": ["Carabidae - Sierra Nevada", "Mycetophilidae", "Aquatic Diptera larvae"]}, status="unknown")
lab = label_fco(LABEL_TEXT or "CA: Alameda Co., Tilden Regional Park\n37.8968, -122.2453  320 m\n12 VI 2019  coll. J. Smith  malaise\nEMEC1005585")
gb = gbif_fco(json.load(open(GBIF_RECORD_JSON)) if GBIF_RECORD_JSON else
              {"key": 0, "catalogNumber": "EMEC1005585", "family": "Carabidae", "order": "Coleoptera", "preparations": "pin", "lifeStage": "Adult"})
fcos = [gb, lab]
if LABEL_PHOTO:
    img = cv2.imread(LABEL_PHOTO)
    fcos.insert(0, vision_label_fco(read_label_with_claude(img), "claude", sha256_bytes(cv2.imencode('.jpg', img)[1].tobytes())))
for f in [home, gaps, *fcos]: print(f"{f.kind:22s} {f.status:9s} id {f.id[:16]}  fields {len(f.fields)}")
ctx = specimen_context(*fcos, gaps=gaps)
print("\ncontext FCO", ctx.id[:16], "parents", [p[:8] for p in ctx.parents])
print("conflicts:", ctx.fields["_conflicts"])
print("missing:", ctx.fields["missing"], "| collection-gap hits:", ctx.fields.get("collection_gap_hits"))
print("\n" + vision_prompt_context(ctx))

essig_page:home        unknown   id 764354caaa69c13e  fields 4
essig_page:gaps        unknown   id a55e6e2695210879  fields 1
gbif_record            observed  id c920d1ddca7a7c9e  fields 6
essig_label            observed  id 45e879f18f9523e4  fields 8

context FCO 1a90f11e3e16623e parents ['c920d1dd', '45e879f1', 'a55e6e26']
conflicts: []
missing: ['scientificName'] | collection-gap hits: ['Carabidae - Sierra Nevada']

Specimen context (FCO 1a90f11e3e16623e):
family: Carabidae
order: Coleoptera
lifeStage: Adult
preparations: pin
method: malaise
locality: CA: Alameda Co., Tilden Regional Park
date: 12 VI 2019
collection_gap_hits: ['Carabidae - Sierra Nevada']
missing: ['scientificName']


### How the context reaches the vision model
* **Grounding**: `vision_prompt_context(ctx)` goes into the Claude QA prompt (`claude_review.review`), so it judges debris and specimens knowing the family, prep and method.
* **Priors**: family → width range (`family_sizes.yaml`) narrows the expected size class before measuring.
* **Novelty**: `gap_novelty_boost` marks specimens from taxa the collection lacks, so they are prioritised for imaging and picking.
* **Fast training**: `bugpicker_bridge.export_yolo(scan, out, class_of=...)` turns BugPicker scans into a YOLO dataset labelled by size class or family from the context FCO; fine-tune `yolo11n` in minutes.

## Test case: novel sizes → mesh gaps → filter suggestions with preservation

In [3]:
from baebugs.sieve import load_yaml
from baebugs.novelty import novel_sizes, mesh_gaps, suggest_filters
from baebugs.preservation import part_risks
from baebugs.explore import Ledger, simulate_run, lomo, compare_lomo_breakpoints, HARDWARE, CAPTURE
rng = np.random.default_rng(SEED); MAT = {m['id']: m for m in load_yaml('materials.yaml')['materials']}
def bag():
    rows=[]
    for n,L,W,p in [(12,(5,9),(2.0,3.0),'hard_bodied'),(10,(8,14),(2.9,3.5),'scaled_wings'),
                    (10,(4,7),(0.8,1.2),'long_appendage'),(14,(1.5,3),(0.7,1.2),'soft_bodied'),(16,(2,5),(0.3,0.6),'membrane_wings')]:
        rows += [dict(length_mm=rng.uniform(*L), width_mm=rng.uniform(*W), profile=p) for _ in range(n)]
    return pd.DataFrame(rows)
sample = pd.read_csv(SAMPLE_CSV) if SAMPLE_CSV else bag()
ref = pd.DataFrame(dict(length_mm=np.r_[rng.uniform(5,8,150), rng.uniform(1.6,2.2,150), rng.uniform(5,9,60), rng.uniform(5,8,40)],
                        width_mm=np.r_[rng.uniform(1.5,2.3,150), rng.uniform(1.0,1.4,150), rng.uniform(4.5,7,60), rng.uniform(2.0,3.0,40)]))
sample['novel'] = [r['novel'] for r in novel_sizes(sample.length_mm.values, sample.width_mm.values, ref.length_mm.values, ref.width_mm.values)]
print(sample.groupby('profile').novel.agg(['sum','count']))
kit = [MAT[i]['aperture_mm'] for i in ['PERF-5','GRID-4','HEX-2','SCREEN-1']]
gap_list = mesh_gaps(sample[sample.novel].width_mm.tolist(), kit)
sugg = {}
for g in gap_list:
    prof = sample[sample.width_mm.between(*g['between_mm'])].profile.value_counts().index[0]
    s = suggest_filters(g, prof, preservation=PRESERVATION); sugg[tuple(g['between_mm'])] = (prof, s)
    print(f"\nGAP {g['between_mm']} mm: {len(g['widths'])} novel, mostly {prof}")
    print(pd.DataFrame(s)[[c for c in ['material','aperture_mm','in_kit','split_quality','retains','worst_part','damage_risk','score','note'] if c in pd.DataFrame(s)]].to_string(index=False))

                sum  count
profile                   
hard_bodied       1     12
long_appendage   10     10
membrane_wings   16     16
scaled_wings      8     10
soft_bodied      11     14

GAP [2.42, 4.4] mm: 9 novel, mostly scaled_wings
      material  aperture_mm  in_kit  split_quality retains worst_part  damage_risk  score
         HEX-2          2.2    True           0.86     9/9      wings        0.248  0.485
PRINT-SMOOTH-3          3.0   False           0.83     3/9      wings        0.207  0.423
        PERF-3          3.0   False           0.83     3/9      wings        0.250  0.358

GAP [0.0, 1.32] mm: 37 novel, mostly membrane_wings
   material  aperture_mm  in_kit  split_quality retains worst_part  damage_risk  score
  NYLON-250         0.25   False           0.80   37/37      wings        0.238  0.340
  NYLON-500         0.50   False           0.51   23/37      wings        0.238  0.053
SS-WOVEN-07         0.71   False           0.16   19/37      wings        0.232 -0.289


In [4]:
mats = ['HEX-2','GRID-4','SCREEN-1','NET-MOSQ','NYLON-500','PRINT-SMOOTH-3']
risk = pd.DataFrame({m: {p: max(part_risks(p, MAT[m], preservation=PRESERVATION).values()) for p in load_yaml('materials.yaml')['specimen_profiles']} for m in mats})
print('worst-part damage risk (default flow):'); print(risk.round(2))

worst-part damage risk (default flow):
                HEX-2  GRID-4  SCREEN-1  NET-MOSQ  NYLON-500  PRINT-SMOOTH-3
hard_bodied      0.32    0.30      0.24      0.18       0.16            0.22
scaled_wings     0.60    0.57      0.47      0.37       0.35            0.45
membrane_wings   0.49    0.47      0.38      0.29       0.27            0.36
long_appendage   0.60    0.57      0.47      0.37       0.35            0.45
soft_bodied      0.56    0.53      0.44      0.34       0.32            0.42


## Explore filters × hardware × capture: LOMO vs Merkle break points

In [5]:
lo, hi = gap_list[0]['between_mm'] if gap_list else (2.4, 4.4)
bodies = [dict(width_mm=w, profile=p) for w, p in zip(sample.width_mm, sample.profile)]
filters = sorted({'GRID-4','HEX-2','SCREEN-1','PERF-5'} | {s['material'] for _, ss in sugg.values() for s in ss if s.get('material') in MAT})
space = [dict(filter=f, hardware=h, capture=c, flow=fl, preservation=PRESERVATION) for f,h,c,fl in itertools.product(filters, HARDWARE, CAPTURE, [0.25,0.5,1.0])]
led = Ledger()
for i, j in enumerate(rng.permutation(len(space))[:120]):
    led.append(space[j], simulate_run(space[j], bodies, (lo, hi), seed=SEED+i))
print('ledger verifies:', led.verify(), '| root:', led.breakpoints[-1]['root'][:16])
print(pd.DataFrame([dict(step=b['step'], root=b['root'][:10], delta=b['delta'], **{k: b['best']['config'][k] for k in ('filter','hardware','capture','flow')}, score=b['best']['score']) for b in led.golden_path()]).to_string(index=False))
for f in ('filter','hardware','capture'): print('\n', pd.DataFrame(lomo(led, f))[['level','n_runs','drop_if_removed']].to_string(index=False))
print('\nLOMO vs break points:', json.dumps({k: {kk: v[kk] for kk in ('lomo_top','lomo_drop','golden_path_final','verdict')} for k, v in compare_lomo_breakpoints(led).items()}, indent=1))
kit_best = max((r for r in led.runs if r['config']['filter'] in ('GRID-4','HEX-2','PERF-5','SCREEN-1')), key=lambda r: r['score'])
best = max(led.runs, key=lambda r: r['score'])
print('\nH0 (kit stack is as good) rejected:', best['score'] - kit_best['score'] > 0.05, '| kit', kit_best['score'], 'vs best', best['score'], best['config'])
print('NOTE: run outcomes are simulated; replace with verify-step counts from the bench.')

ledger verifies: True | root: d1bae5c2f1106a54
 step       root  delta    filter        hardware          capture  flow  score
    1 0363fbc15d 0.1169 NYLON-250    partabot_arm   amscope_md300a  0.25 0.1169
    6 aa1dfebcd6 0.0121  SCREEN-1    partabot_arm   amscope_md300a  0.50 0.1290
   26 da31a85010 0.0126 NYLON-250 esp32_bugsorter   amscope_md300a  0.25 0.1416
   33 3e2f15340b 0.0040 NYLON-500    partabot_arm            phone  0.25 0.1456
   52 fea45e30b6 0.0295 NYLON-500          manual   amscope_md300a  0.25 0.1751
   57 0aaab25292 0.0920     HEX-2 esp32_bugsorter bugpicker_topcam  0.25 0.2671
   72 b6343d2f9c 0.1195  SCREEN-1          manual   amscope_md300a  0.25 0.3866

          level  n_runs  drop_if_removed
      SCREEN-1      15           0.0084
        GRID-4      14           0.0000
         HEX-2      14           0.0000
     NYLON-250      11           0.0000
     NYLON-500      15           0.0000
        PERF-3      11           0.0000
        PERF-5      10         